# 19 — NeMo Guardrails with the same Ollama model
Input and output self-check rails are advisory safety screens. Pydantic,
citation checks, and server-side approval remain the hard boundaries.



## Problem

A model can receive hostile instructions or return an unsafe draft.


## Concept

NeMo self-check rails use the configured Ollama model as an opt-in screen.

NeMo checks input/output without replacing CrewAI generation. Its Ollama engine uses the configured local model.


## Abstraction

question → NeMo input check → CrewAI → NeMo output check → policy gate. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
input → NeMo/Ollama input rail → CrewAI → NeMo output rail → Python approval gate
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Inspect the shared Ollama configuration

Inspect the shared Ollama configuration.


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.nemo_guardrails import check_input, check_output, enabled

print("NeMo enabled:", enabled())
print("Configured model:", os.getenv("WORKSHOP_MODEL", "ollama/qwen3.5:4b"))



## Run a local-model check when NeMo is enabled
This is opt-in because the self-check invokes the local model twice.



## Step 2 — Run input and output checks only when the local model is ready

Run input and output checks only when the local model is ready.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    if not enabled():
        raise RuntimeError("Set WORKSHOP_NEMO_ENABLED=true for the live guardrail lab")
    question = "Should Northstar AI pilot Meridian AI?"
    check_input(question)
    check_output(question, '{"recommendation":"pilot","rationale":"verify security claims"}')
    print("Input and output rails passed")
else:
    print("Enable NeMo and WORKSHOP_RUN_LIVE_DEMOS=true for model-backed checks.")


## Execution flow

question → NeMo input check → CrewAI → NeMo output check → policy gate. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

A self-check can miss an attack or block a benign question. The model never authorizes a side effect.


## Production implication

Pin and evaluate NeMo on the target local model; define timeouts and retain hard policy checks.


## Your turn

Try a prompt-injection fixture and inspect the check result.

**Checkpoint:** Model-backed rails are visible and never replace deterministic authorization.
